For reference use Luis' notebook https://notebooksharing.space/view/a0214376374af43776a54bf69b0b0308bbc0499fad345fcb5eb70e9ad9cf46c6#displayOptions=

In [3]:
import earthaccess as ea
import xarray as xr
import zarr
from urllib.parse import urlparse
import fsspec
import warnings
import logging

import dask
from dask.distributed import Client, LocalCluster
from coiled import Cluster as CoiledCluster
import virtualizarr as vz

import obstore
from obstore.auth.earthdata import NasaEarthdataCredentialProvider # Only S3 links access in-region
from obstore.store import HTTPStore, S3Store
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import DMRPPParser, NetCDF3Parser, HDFParser

for _ in (ea, vz, zarr, xr, obstore, dask, fsspec):
    print(f"{_.__name__}: {_.__version__}")

earthaccess: 0.16.0
virtualizarr: 2.4.0
zarr: 3.1.5
xarray: 2025.6.1
obstore: 0.8.2
dask: 2024.5.2
fsspec: 2025.7.0


In [4]:
auth = ea.login()

Enter your Earthdata Login username:  deanh808
Enter your Earthdata password:  ········


## 1. Setup Local Dask Cluster

In [5]:
def create_dask_cluster(n_workers):
    import logging
    print("Creating new local Dask client")
    cluster = LocalCluster(
        n_workers=n_workers,
        threads_per_worker=1,
        silence_logs=logging.ERROR) 

    client = Client(cluster)
    return (client, cluster)


def silence_worker_warnings_and_auth(token):
    import warnings
    import logging
    import earthaccess as ea  # local instance of earthaccess is explicitely authenticated
    import os

    if token:
        os.environ["EARTHDATA_TOKEN"]= token
        auth = ea.login(strategy="environment")
    
    warnings.filterwarnings("ignore")
    for name in ["distributed", "xarray", "py.warnings", "fsspec", "h5netcdf", "h5py"]:
        logging.getLogger(name).setLevel(logging.ERROR)

In [7]:
n_workers = 4

if "client" not in locals() or "cluster" not in locals():
    client, cluster = create_dask_cluster(n_workers = n_workers)
client

Creating new local Dask client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-cjsrz.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-cjsrz.dask.host/jupyter/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 14.50 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36247,Workers: 4
Dashboard: https://cluster-cjsrz.dask.host/jupyter/proxy/8787/status,Total threads: 4
Started: Just now,Total memory: 14.50 GiB
Comm: tcp://127.0.0.1:35515,Total threads: 1
Dashboard: https://cluster-cjsrz.dask.host/jupyter/proxy/43243/status,Memory: 3.63 GiB
Nanny: tcp://127.0.0.1:33505,


In [8]:
%%capture
client.run(silence_worker_warnings_and_auth, auth.token["access_token"])

## 2. Single file reference generation

In [ ]:
def create_single_refs_s3(granule_info, 
                          credentials_url = "https://archive.podaac.earthdata.nasa.gov/s3credentials",
                          bucket = "s3://podaac-ops-cumulus-protected",
                          region = "us_west-2"):

    ## 1. Define wrapper around virtualizarr.open_virtual_dataset() but with a specific warning supressed
    def open_vds_warnsupressed(datalink, **kwargs):
    """
    Just a wrapper around virtualizarr.open_virtual_dataset() but with a specific warning supressed.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Numcodecs codecs*", category=UserWarning)
        vds_ref = open_virtual_dataset(
                datalink, **kwargs
            )
    return vds_ref


    ## 2. Create VDS object store with needed creds
    parsed_granule_url = urlparse(granule_info[0].data_links(access="direct")[0])
    bucket = parsed_granule_url.netloc

    s3_store = S3Store(
        bucket = bucket,
        region = region,
        credential_provider = NasaEarthdataCredentialProvider(credentials_url),
        virtual_hosted_style_request = False,
        client_options={"allow_http": True},
    )
    s3_obstore_registry = ObjectStoreRegistry({f"s3://{bucket}": s3_store})


    

In [10]:
granule_info = ea.search_data(
        short_name= "SASSIE_ECCO_L4_OBP_SSH_LLC1080GRID_SNAPSHOT_V1R1",
    )

In [5]:
credential_provider = NasaEarthdataCredentialProvider(credentials_url, auth=(auth.username, auth.password))
store = S3Store.from_url(bucket, credential_provider=credential_provider)
registry = ObjectStoreRegistry({bucket: store})